# 262. Trips and Users
https://leetcode.com/problems/trips-and-users/

## Complexity Comparison

| Approach | Time | Space | Description |
|----------|------|-------|-------------|
| Subquery filter | O(n*m) | O(n) | Filter unbanned, group by day |
| **JOIN + GROUP BY (Optimal)** | **O(n)** | **O(n)** | Join with users, filter, aggregate |

## Methodology

This is a SQL problem. We join the Trips table with the Users table (for both client and driver) to exclude banned users, filter by the date range, then group by day to calculate the cancellation rate as the ratio of cancelled trips to total trips, rounded to 2 decimal places. The programmatic equivalents in Python/Go/Rust simulate this logic.

## Solutions

### C#

In [ ]:
// SQL Solution (this is a SQL problem on LeetCode)
//
// SELECT t.request_at AS Day,
//        ROUND(
//            SUM(CASE WHEN t.status != 'completed' THEN 1 ELSE 0 END) /
//            COUNT(*),
//            2
//        ) AS 'Cancellation Rate'
// FROM Trips t
// JOIN Users u1 ON t.client_id = u1.users_id AND u1.banned = 'No'
// JOIN Users u2 ON t.driver_id = u2.users_id AND u2.banned = 'No'
// WHERE t.request_at BETWEEN '2013-10-01' AND '2013-10-03'
// GROUP BY t.request_at
// ORDER BY t.request_at;

### Python

In [ ]:
import pandas as pd

def trips_and_users(trips: pd.DataFrame, users: pd.DataFrame) -> pd.DataFrame:
    banned = set(users[users['banned'] == 'Yes']['users_id'])
    filtered = trips[
        (~trips['client_id'].isin(banned)) &
        (~trips['driver_id'].isin(banned)) &
        (trips['request_at'] >= '2013-10-01') &
        (trips['request_at'] <= '2013-10-03')
    ]
    if filtered.empty:
        return pd.DataFrame(columns=['Day', 'Cancellation Rate'])
    result = filtered.groupby('request_at').apply(
        lambda g: round(sum(g['status'] != 'completed') / len(g), 2)
    ).reset_index()
    result.columns = ['Day', 'Cancellation Rate']
    return result

### Go

In [ ]:
type Trip struct {
    Id        int
    ClientId  int
    DriverId  int
    Status    string
    RequestAt string
}

type User struct {
    UsersId int
    Banned  string
}

func tripsAndUsers(trips []Trip, users []User) map[string]float64 {
    banned := make(map[int]bool)
    for _, u := range users {
        if u.Banned == "Yes" {
            banned[u.UsersId] = true
        }
    }
    total := make(map[string]int)
    cancelled := make(map[string]int)
    for _, t := range trips {
        if banned[t.ClientId] || banned[t.DriverId] {
            continue
        }
        if t.RequestAt >= "2013-10-01" && t.RequestAt <= "2013-10-03" {
            total[t.RequestAt]++
            if t.Status != "completed" {
                cancelled[t.RequestAt]++
            }
        }
    }
    result := make(map[string]float64)
    for day, tot := range total {
        result[day] = math.Round(float64(cancelled[day])/float64(tot)*100) / 100
    }
    return result
}

### Rust

In [ ]:
use std::collections::{HashMap, HashSet};

struct Trip {
    client_id: i32,
    driver_id: i32,
    status: String,
    request_at: String,
}

struct User {
    users_id: i32,
    banned: String,
}

fn trips_and_users(trips: &[Trip], users: &[User]) -> HashMap<String, f64> {
    let banned: HashSet<i32> = users.iter()
        .filter(|u| u.banned == "Yes")
        .map(|u| u.users_id)
        .collect();
    let mut total: HashMap<String, i32> = HashMap::new();
    let mut cancelled: HashMap<String, i32> = HashMap::new();
    for t in trips {
        if banned.contains(&t.client_id) || banned.contains(&t.driver_id) {
            continue;
        }
        if t.request_at >= "2013-10-01" && t.request_at <= "2013-10-03" {
            *total.entry(t.request_at.clone()).or_insert(0) += 1;
            if t.status != "completed" {
                *cancelled.entry(t.request_at.clone()).or_insert(0) += 1;
            }
        }
    }
    total.iter().map(|(day, &tot)| {
        let canc = *cancelled.get(day).unwrap_or(&0) as f64;
        (day.clone(), (canc / tot as f64 * 100.0).round() / 100.0)
    }).collect()
}

## Example Scenarios

1. **Day 2013-10-01** - 4 trips (after filtering banned), 1 cancelled by client. Rate = 1/4 = 0.33.

2. **Day 2013-10-02** - 3 trips, 0 cancelled. Rate = 0/3 = 0.00.

3. **Day 2013-10-03** - 2 trips, 1 cancelled by driver. Rate = 1/2 = 0.50.

4. **Banned user trips** - If client_id=2 is banned, all trips by user 2 are excluded from calculations.

5. **No trips on a day** - If all trips on a day involve banned users, that day does not appear in results.

![image](attachment:image.png)